# Legal RAG Hackathon: Main Colab

Один ноутбук для одноразового setup и повторяемого цикла: запустить нужный эксперимент, посмотреть метрики, при необходимости собрать submission.

## 1. Setup

Этот блок обычно выполняется один раз после старта или рестарта Colab.

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/Timur713/Legal_RAG_HSE"
REPO_DIR = Path("/content/legal-rag-hackathon")
COLAB_PATHS = "configs/paths.colab.yaml"

print(f"Repo dir: {REPO_DIR}")
print(f"Paths config: {COLAB_PATHS}")

In [ ]:
%cd /content

if not REPO_DIR.exists():
    if REPO_URL == "<PASTE_REPO_URL_HERE>":
        raise ValueError("Set REPO_URL before cloning the repository.")
    repo_name = REPO_DIR.name
    !git clone $REPO_URL $repo_name

%cd $REPO_DIR
!git pull

In [ ]:
%cd $REPO_DIR
!pip install -r requirements.txt

In [ ]:
%cd $REPO_DIR
!python scripts/check_data.py --paths $COLAB_PATHS

### 1.1 Prepare Validation Splits

Фиксированные локальные сплиты для `strict_cv`, `strict_holdout` и вспомогательного `relaxed_cv` готовим один раз в setup.

In [ ]:
VALIDATION_SPLITS_CMD = f"python scripts/make_validation_splits.py --paths {COLAB_PATHS}"

print(VALIDATION_SPLITS_CMD)

In [ ]:
%cd $REPO_DIR
!$VALIDATION_SPLITS_CMD

## 2. Run Experiment + View Metrics

Меняйте параметры ниже для каждого нового эксперимента. Этот блок можно повторять сколько угодно раз.

### 2.1 Experiment: Baseline

Первый эксперимент: исходный baseline без локальной validation-обвязки.

In [ ]:
EXPERIMENT_NAME = "baseline"
EXPERIMENT_CMD = f"python scripts/run_baseline.py --paths {COLAB_PATHS} --experiment-name {EXPERIMENT_NAME}"

print(EXPERIMENT_CMD)

In [ ]:
%cd $REPO_DIR
!$EXPERIMENT_CMD

### 2.2 Experiment: Baseline Validation

Запускаем baseline не по всему train сразу, а с учетом локального validation-протокола: `strict_cv_mean`, `strict_cv_std` и `strict_holdout`.

In [ ]:
EXPERIMENT_NAME = "baseline_validation"
EXPERIMENT_CMD = (
    f"python scripts/run_baseline_validation.py "
    f"--paths {COLAB_PATHS} "
    f"--experiment-name {EXPERIMENT_NAME} "
    f"--save-test-predictions"
)

print(EXPERIMENT_CMD)

In [ ]:
%cd $REPO_DIR
!$EXPERIMENT_CMD

### 2.3 View Metrics

Смотрим метрики текущего эксперимента из сохраненного JSON.

In [ ]:
from pathlib import Path
import json

metrics_path = REPO_DIR / "outputs" / "metrics" / f"{EXPERIMENT_NAME}_metrics.json"
print(metrics_path)

if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    print(json.dumps(metrics, ensure_ascii=False, indent=2))
else:
    print("Metrics file not found. Check experiment command or output naming.")

## 3. Make Submission If Needed

Запускайте этот блок только для тех экспериментов, для которых нужен сабмит. Обычно он использует `EXPERIMENT_NAME` из секции выше.

In [ ]:
SUBMISSION_CMD = f"python scripts/make_submission.py --paths {COLAB_PATHS} --experiment-name {EXPERIMENT_NAME}"

print(SUBMISSION_CMD)

In [ ]:
%cd $REPO_DIR
!$SUBMISSION_CMD

In [ ]:
submission_path = REPO_DIR / "outputs" / "submissions" / f"{EXPERIMENT_NAME}_submission.csv"
print(submission_path)

if submission_path.exists():
    import pandas as pd
    submission_df = pd.read_csv(submission_path)
    display(submission_df.head())
    print(f"rows={len(submission_df)} qids={submission_df['qid'].nunique()}")
else:
    print("Submission file not found.")

## Optional: Inspect or Save Outputs

Полезно после серии запусков, но не обязательно для каждого эксперимента.

In [ ]:
%cd $REPO_DIR
!find outputs -maxdepth 2 -type f | sort

In [ ]:
%cd $REPO_DIR
!git status

tracked_paths = ["outputs", "experiments"]
status = subprocess.run(
    ["git", "status", "--porcelain", "--", *tracked_paths],
    capture_output=True,
    text=True,
    check=False,
)

if status.stdout.strip():
    subprocess.run(["git", "add", *tracked_paths], check=True)
    subprocess.run(["git", "commit", "-m", f"add outputs for {EXPERIMENT_NAME}"], check=False)
    subprocess.run(["git", "push"], check=False)
else:
    print("Nothing new to commit in outputs/ or experiments/.")